<a href="https://colab.research.google.com/github/DivyaS-617/AD-Drug-Insights-RAG/blob/main/Bootcampmar26/Day_2_rag_llamaindex_mar26-pdf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install llama_index

In [ ]:
import openai
from google.colab import userdata

# Retrieve the OpenAI API key from Google Colab secrets
openai.api_key = userdata.get('openai')

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PDFReader

documents = SimpleDirectoryReader(
    input_dir="data_new",
    required_exts=[".pdf"],
    file_extractor={".pdf": PDFReader()}
).load_data()

print(len(documents))
print(documents[0].text[:1000])

9
DDS Employee Handbook (Synthetic) v1
Effective date: March 03, 2026  Dubai (GST)
Note: This document is a synthetic, training-friendly employee handbook for demos, onboarding
simulations, and HR-policy chatbot prototypes. It is not legal advice and must be reviewed by qualified
counsel before any real-world use.
1. Welcome to Decoding Data Science (DDS)
DDS is a Dubai-based academy, consulting practice, and community focused on data science, AI, and
applied generative AI. We operate with a global mindset and a high trust culture—shipping practical
outcomes while supporting each other.
This handbook explains workplace expectations, benefits, and policies. If any local law conflicts with
this handbook, applicable law prevails.
2. Company Values & Ways of Working
 Build with clarity: define the user, problem, inputs/outputs, and definition of done.
 Bias for action: ship small, iterate fast, measure outcomes.
 Respect and inclusion: disagreement is allowed; disrespect is not.
 Data

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
documents = SimpleDirectoryReader("data_new").load_data()
index = VectorStoreIndex.from_documents(documents=documents)
query_engine = index.as_query_engine()
response = query_engine.query("how to take sick leave?")
print(response)

To take sick leave, notify your manager as soon as possible when you are unwell and unable to work. If your absence lasts for 2 or more consecutive workdays, a medical certificate may be required. If you are able to work partially, discuss this arrangement with your manager, prioritizing your health.


In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Configure LLM, Embedding, and Chunk Size
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

# Define a system prompt
system_prompt = '''

You are Ayesha, the Decoding Data Science (DDS) Enterprise HR Chatbot. Your objective is to interact politely and professionally with employees, answering only HR-related questions. Use only information directly from the 4 connected HR documents to provide your answers. Always provide an explicit citation indicating the document source for every answer. Do not offer information or suggestions beyond what is present in these documents.

If the requested information cannot be found in the connected documents, politely instruct the user to email connect@decodingdatascience.com for further assistance. For questions outside of HR (such as food, restaurants, or non-work matters), inform the user that you can only answer HR-related questions. If a question is unclear or possibly HR-related but ambiguous, persist in seeking clarification or ask the user to rephrase. Never attempt to answer non-HR, personal, or unrelated questions.

- Remain polite and professional in all interactions.
- Respond exclusively to HR-related topics (e.g., payroll, benefits, time-off, HR policies, leave, compliance, hiring, employee development) using only the connected HR documents.
- Always include an explicit citation to the relevant document(s) for every answer.
- If a question is off-topic, state that you can only address HR questions.
- If you are unsure whether the question is HR-related, politely ask for clarification or rephrasing.
- For HR-related questions where the answer is not in the connected documents, kindly direct the user to email connect@decodingdatascience.com.
- Use a friendly and formal tone.
- Always persist in seeking clarification for ambiguous or insufficiently clear questions before providing an answer or deferring.

**Output Format:**
Respond in short, clear paragraphs (2-4 sentences). Do not use markdown or code blocks. Every answer must include the citation showing which document(s) the information is sourced from.

# Steps

1. Determine if the question is clearly HR-related.
2. If not HR-related, politely inform the user of your scope concerning only HR queries.
3. If the question is ambiguous or lacks clarity, persistently request clarification or rephrasing.
4. For HR questions, search the connected 4 HR documents.
    - If the answer is found, provide it with an explicit citation (e.g., "Document 2: Leave Policy, Section 3").
    - If the answer is not found in the documents, politely request the user to email connect@decodingdatascience.com for more assistance.

# Output Format

Respond in prose (2-4 clear sentences), including an explicit citation for every answer (e.g., "Source: Document 2, Leave Policy"). Do not use markdown or code blocks.

# Examples

**Example 1**
Input: What is the process for applying for maternity leave?
Output: Thank you for your question regarding maternity leave. To apply for maternity leave, please fill out the leave request form available on the HR portal and submit it to your manager for approval. Source: Document 2, Leave Policy.

**Example 2**
Input: Where is the best place to get lunch nearby?
Output: I can only assist with HR-related questions. If you have a question about HR policies, benefits, or company leave, please let me know!

**Example 3**
Input: Can you tell me about the company gym facilities?
Output: I am here to help with HR-related queries only. If your question relates to your employment, benefits, or HR policies, please clarify or rephrase your question.

**Example 4**
Input: How do I update my emergency contact information?
Output: You can update your emergency contact information by logging into the HR portal and editing your profile details. Source: Document 4, Employee Handbook.

**Example 5**
Input: How can I apply for a sabbatical leave?
Output: I am sorry, but I could not find information regarding sabbatical leave in our HR documents. For further assistance, please email connect@decodingdatascience.com.

**Important Instructions Reminder:**
- Answer HR questions strictly from the 4 connected documents and always provide a citation.
- For topics not covered in the documents, direct users to email connect@decodingdatascience.com.
- For off-topic or unclear queries, request clarification or state your HR-only scope.
- Remain friendly, professional, and persistent in seeking clarification when needed.


'''

documents = SimpleDirectoryReader("data").load_data()

index = VectorStoreIndex.from_documents(documents=documents)

# Configure query engine with system prompt
query_engine = index.as_query_engine(system_prompt=system_prompt)

response = query_engine.query("What are DDS standard office hours in Dubai?")
print(response)

The standard office hours in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.


In [ ]:
documents

[Document(id_='d188dd81-a11c-45c3-aa18-64e251d634a5', embedding=None, metadata={'file_path': '/content/data/DDS Employee Handbook.txt', 'file_name': 'DDS Employee Handbook.txt', 'file_type': 'text/plain', 'file_size': 8080, 'creation_date': '2026-03-28', 'last_modified_date': '2026-03-28'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text="DDS Employee Handbook (Synthetic) v1\r\nEffective date: March 03, 2026  Dubai (GST)\r\nNote: This document is a synthetic, training-friendly employee handbook for demos, onboarding\r\nsimulations, and HR-policy chatbot prototypes. It is not legal advice and must be reviewed by qualified\r\ncounsel before a

In [ ]:
import time
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# Start timer
start_time = time.time()

# Load and index documents
documents = SimpleDirectoryReader("data").load_data()
index = VectorStoreIndex.from_documents(documents=documents)

# Query the index
query_engine = index.as_query_engine()
response = query_engine.query("What are DDS standard office hours in Dubai?")
print(response)

# End timer and print duration
end_time = time.time()
print(f"\nExecution Time: {end_time - start_time:.2f} seconds")


The standard office hours in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.

Execution Time: 2.56 seconds


In [ ]:
pip install gradio

In [ ]:
import gradio as gr
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# Configure LLM, Embedding, and Chunk Size
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

# Load data and build the index
documents = SimpleDirectoryReader("data").load_data()
index = VectorStoreIndex.from_documents(documents=documents)
query_engine = index.as_query_engine()

# Function to handle queries
def query_document(query):
    response = query_engine.query(query)
    return str(response)

# Gradio interface
interface = gr.Interface(
    fn=query_document,
    inputs=gr.Textbox(label="Enter your query", placeholder="Type your question here..."),
    outputs=gr.Textbox(label="Response"),
    title="DDS Enterise Chatbot connect to Data",
    description="Ask questions about the documents loaded into the system."
)

# Launch the Gradio app
if __name__ == "__main__":
    interface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ac7f995d87bb8ebf4c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

# check if storage already exists
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

# Either way we can now query the index
query_engine = index.as_query_engine()

retriever = VectorIndexRetriever(index=index, similarity_top_k=3)

query_engine = RetrieverQueryEngine(retriever=retriever)

response = query_engine.query("What are DDS standard office hours in Dubai?")
print(response)


The standard office hours in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.


In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PDFReader

documents = SimpleDirectoryReader(
    input_dir="data_new",
    required_exts=[".pdf"],
    file_extractor={".pdf": PDFReader()}
).load_data()

print(len(documents))
print(documents[0].text[:1000])

9
DDS Employee Handbook (Synthetic) v1
Effective date: March 03, 2026  Dubai (GST)
Note: This document is a synthetic, training-friendly employee handbook for demos, onboarding
simulations, and HR-policy chatbot prototypes. It is not legal advice and must be reviewed by qualified
counsel before any real-world use.
1. Welcome to Decoding Data Science (DDS)
DDS is a Dubai-based academy, consulting practice, and community focused on data science, AI, and
applied generative AI. We operate with a global mindset and a high trust culture—shipping practical
outcomes while supporting each other.
This handbook explains workplace expectations, benefits, and policies. If any local law conflicts with
this handbook, applicable law prevails.
2. Company Values & Ways of Working
 Build with clarity: define the user, problem, inputs/outputs, and definition of done.
 Bias for action: ship small, iterate fast, measure outcomes.
 Respect and inclusion: disagreement is allowed; disrespect is not.
 Data

In [ ]:
import os
import time
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

# Start timer for index setup
start_time = time.time()

# check if storage already exists
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

setup_duration = time.time() - start_time
print(f"Index setup time: {setup_duration:.2f} seconds")

# Start timer for query
query_start_time = time.time()

# Prepare the query engine
retriever = VectorIndexRetriever(index=index, similarity_top_k=2)
query_engine = RetrieverQueryEngine(retriever=retriever)

# Execute query
response = query_engine.query("Who all mentioned in the doc?")
print(response)

query_duration = time.time() - query_start_time
print(f"Query time: {query_duration:.2f} seconds")


Index setup time: 0.10 seconds
The document does not mention specific individuals by name. It outlines policies and procedures related to various topics such as communication, workplace safety, disciplinary processes, and more, but does not provide any personal names or identifiers.
Query time: 1.48 seconds


In [ ]:
import os
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

import time
start_time = time.time()
query_engine = index.as_query_engine()

retriever = VectorIndexRetriever(index=index, similarity_top_k=2)

query_engine = RetrieverQueryEngine(retriever=retriever)

response = query_engine.query("What are DDS standard office hours in Dubai??")
print(response)


end_time = time.time()  # Record end time
execution_time = end_time - start_time  # Calculate execution time
print(f"Execution time: {execution_time} seconds")

The standard office hours in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.
Execution time: 1.9070744514465332 seconds
